In [1]:
import pandas as pd
import numpy as np

# Cleaning Nav History

In [2]:
#importing data
df_nav_history = pd.read_csv('../data/raw/02_nav_history.csv')

In [4]:
df_nav_history.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [6]:
df_nav_history.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  object 
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ MB


In [7]:
df_nav_history['date'] = pd.to_datetime(df_nav_history['date'], format='%Y-%m-%d')

In [11]:
df_nav_history.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.1 MB


In [13]:
print("Duplicated rows: ", df_nav_history.duplicated().sum())
print("Missing values: ", df_nav_history.isnull().sum())

Duplicated rows:  0
Missing values:  amfi_code    0
date         0
nav          0
dtype: int64


In [14]:
df_nav_history = df_nav_history.sort_values(by=['amfi_code', 'date'], ascending=[True, True])

In [15]:
df_nav_history.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [16]:
#fill missing values 
df_nav_history = (df_nav_history.set_index("date").groupby("amfi_code").apply(lambda x: x.asfreq('D').ffill()).reset_index(level=0, drop=True).reset_index())


C:\Users\anshy\AppData\Local\Temp\ipykernel_32328\249137861.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_nav_history = (df_nav_history.set_index("date").groupby("amfi_code").apply(lambda x: x.asfreq('D').ffill()).reset_index(level=0, drop=True).reset_index())


In [22]:
df_nav_history['amfi_code'] = df_nav_history['amfi_code'].astype(int)

In [23]:
df_nav_history.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64320 entries, 0 to 64319
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       64320 non-null  datetime64[ns]
 1   amfi_code  64320 non-null  int32         
 2   nav        64320 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int32(1)
memory usage: 1.2 MB


In [24]:
df_nav_history.head()

,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639


In [20]:
print("Duplicated rows:", df_nav_history.duplicated().sum())
print("Missing values:", df_nav_history.isnull().sum())

Duplicated rows: 0
Missing values: date         0
amfi_code    0
nav          0
dtype: int64


In [25]:
if (df_nav_history["nav"] > 0).all():
    print("✅ All NAV values are valid.")
else:
    print("❌ Dataset contains invalid NAV values.")

✅ All NAV values are valid.


In [65]:
df_nav_history["daily_return_pct"] = (df_nav_history.groupby("amfi_code")["nav"].pct_change()*100)

In [67]:
df_nav_history.head()

,date,amfi_code,nav,daily_return_pct
0,2022-01-03,100016,520.4608,NaN
1,2022-01-04,100016,515.0971,-1.030568
2,2022-01-05,100016,521.7239,1.286515
3,2022-01-06,100016,515.7880,-1.137747
4,2022-01-07,100016,515.1639,-0.120999


In [68]:
#Saving the dataset
df_nav_history.to_csv('../data/processed/02_nav_history_cleaned.csv', index=False)

# Cleaning Investor Transaction

In [33]:
df_it = pd.read_csv('../data/raw/08_investor_transactions.csv')

In [43]:
df_it.sample(5)

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
24036,INV002393,2025-01-13,101208,SIP,448,Madhya Pradesh,Bhopal,B30,26-35,Female,15.2,UPI,Verified
10775,INV002349,2024-06-18,118634,SIP,3109,Punjab,Amritsar,B30,26-35,Female,9.6,Mandate,Verified
8512,INV003560,2024-05-14,119598,Redemption,418214,Rajasthan,Udaipur,B30,26-35,Male,22.5,Mandate,Verified
8020,INV004896,2024-05-05,119552,Redemption,12392,Uttar Pradesh,Agra,B30,18-25,Female,3.5,Net Banking,Verified
8024,INV004701,2024-05-05,118632,SIP,14454,Haryana,Gurugram,T30,26-35,Male,7.3,Net Banking,Verified


In [35]:
df_it.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB


In [39]:
df_it['transaction_type'].value_counts()

transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

In [40]:
mapping = {
    "sip": "SIP",
    "Sip": "SIP",
    "S.I.P": "SIP",
    "Systematic Investment Plan": "SIP",
    "Lump Sum": "Lumpsum",
    "lumpsum": "Lumpsum",
    "redeem": "Redemption",
    "REDEMPTION": "Redemption"
}

df_it["transaction_type"] = df_it["transaction_type"].replace(mapping)

In [41]:
df_it['transaction_type'].value_counts()

transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

In [42]:
if (df_it['amount_inr'] > 0).all():
    print("✅ All amount_inr values are valid.")
else:
    print("❌ Dataset contains invalid amount_inr values.")

✅ All amount_inr values are valid.


In [44]:
df_it['transaction_date'] = pd.to_datetime(df_it['transaction_date'], format='%Y-%m-%d')

In [45]:
df_it.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   investor_id         32778 non-null  object        
 1   transaction_date    32778 non-null  datetime64[ns]
 2   amfi_code           32778 non-null  int64         
 3   transaction_type    32778 non-null  object        
 4   amount_inr          32778 non-null  int64         
 5   state               32778 non-null  object        
 6   city                32778 non-null  object        
 7   city_tier           32778 non-null  object        
 8   age_group           32778 non-null  object        
 9   gender              32778 non-null  object        
 10  annual_income_lakh  32778 non-null  float64       
 11  payment_mode        32778 non-null  object        
 12  kyc_status          32778 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), ob

In [46]:
df_it['kyc_status'].value_counts()

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

In [47]:
print("Duplicate rows:", df_it.duplicated().sum())
print("Missing values:", df_it.isnull().sum())

Duplicate rows: 0
Missing values: investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64


In [69]:
df_it["tx_id"] = range(1, len(df_it) + 1)

In [92]:
df_it.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status,tx_id
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified,1
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified,2
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified,3
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending,4
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending,5


In [93]:
df_it.to_csv('../data/processed/08_investor_transactions_cleaned.csv', index=False)

# Cleaning Scheme Performance

In [49]:
df_sp = pd.read_csv('../data/raw/07_scheme_performance.csv')

In [50]:
df_sp.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [51]:
df_sp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

In [52]:
return_cols = ["return_1yr_pct", "return_3yr_pct", "return_5yr_pct"]

for col in return_cols:
    df_sp[col] = pd.to_numeric(df_sp[col], errors='coerce')

In [53]:
invalid_returns = df_sp[df_sp[return_cols].isna().any(axis=1)]

In [54]:
if(invalid_returns.empty):
    print("✅ All return values are valid.")
else:
    print("❌ Dataset contains invalid return values.")
    print(invalid_returns)

✅ All return values are valid.


In [57]:
anomalies = df_sp[(df_sp['return_1yr_pct'] < -100) | (df_sp['return_3yr_pct'] < -100) | (df_sp['return_5yr_pct'] < -100) | (df_sp['return_1yr_pct'] > 300) | (df_sp['return_3yr_pct'] > 300) | (df_sp['return_5yr_pct'] > 300)]

In [58]:
if(anomalies.empty):
    print("✅ No anomalies found in return values.")
else:
    print("❌ Anomalies found in return values.")
    print(anomalies)

✅ No anomalies found in return values.


In [59]:
invalid_expense = df_sp[(df_sp['expense_ratio_pct'] < 0.1) | (df_sp['expense_ratio_pct'] > 2.5)]

In [60]:
if(invalid_expense.empty):
    print("✅ All expense ratio values are valid.")
else:
    print("❌ Dataset contains invalid expense ratio values.")
    print(invalid_expense)

✅ All expense ratio values are valid.


In [61]:
df_sp.to_csv('../data/processed/07_scheme_performance_cleaned.csv', index=False)

In [62]:
from pathlib import Path

raw_path = Path("../data/raw")
processed_path = Path("../data/processed")

processed_path.mkdir(exist_ok=True)

files = [
    "01_fund_master.csv",
    "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv"
]

for file in files:
    try:
        df = pd.read_csv(raw_path / file)

        df.columns = df.columns.str.strip()

        df = df.drop_duplicates()

        for col in df.columns:
            if df[col].dtype == "object":
                df[col] = df[col].astype(str).str.strip()

        output_file = processed_path / f"{Path(file).stem}_cleaned.csv"
        df.to_csv(output_file, index=False)

        print(f"Cleaned: {file}")

    except Exception as e:
        print(f"Error in {file}: {e}")

print("Day 2: Data cleaning completed.")

Cleaned: 01_fund_master.csv
Cleaned: 03_aum_by_fund_house.csv
Cleaned: 04_monthly_sip_inflows.csv
Cleaned: 05_category_inflows.csv
Cleaned: 06_industry_folio_count.csv
Cleaned: 09_portfolio_holdings.csv
Cleaned: 10_benchmark_indices.csv
Day 2: Data cleaning completed.


# Creating SQLite Database

In [63]:
import sqlite3

conn = sqlite3.connect('../data/db/bluestock_mf.db')

conn.close()

In [74]:
conn = sqlite3.connect('../data/db/bluestock_mf.db')

with open("../sql/schema.sql", "r") as f:
    conn.executescript(f.read())

conn.commit()
conn.close()

In [75]:
#creating engine

from sqlalchemy import create_engine

engine = create_engine('sqlite:///../data/db/bluestock_mf.db')

In [77]:
#dim_fund

df = pd.read_csv('../data/processed/01_fund_master_cleaned.csv')

df = df[["amfi_code", "fund_house", "scheme_name", "category", "sub_category", "benchmark", "expense_ratio_pct"]]

df.to_sql("dim_fund", engine, if_exists="append", index=False)

40

In [78]:
#dim_date 

dates = pd.date_range(start="2019-01-01", end="2026-06-30", freq='D')

dim_date = pd.DataFrame({"date_id": range(1, len(dates)+1), "date": dates})

dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["day"] = dim_date["date"].dt.day
dim_date["weekday"] = dim_date["date"].dt.dayofweek      # Monday=0
dim_date["is_weekday"] = (dim_date["weekday"] < 5).astype(int)

dim_date.to_sql("dim_date", engine, if_exists="append", index=False)

2738

In [ ]:
#fact_nav

df = pd.read_csv('../data/processed/02_nav_history_cleaned.csv')

df.to_sql("fact_nav", engine, if_exists="append", index=False)

64320

In [ ]:
#fact_performance

df = pd.read_csv("../data/processed/07_scheme_performance_cleaned.csv")

df["as_of_date"] = pd.Timestamp.today().normalize()

df = df[["amfi_code", "as_of_date", "return_1yr_pct", "return_3yr_pct", "return_5yr_pct", "sharpe_ratio", "alpha", "beta", "max_drawdown_pct"]]

df.to_sql("fact_performance", engine, if_exists="append", index=False)

40

In [ ]:
#fact_portfolio

df = pd.read_csv("../data/processed/09_portfolio_holdings_cleaned.csv")

df =df[["amfi_code", "stock_symbol", "stock_name", "sector", "weight_pct", "portfolio_date"]]

df.to_sql("fact_portfolio", engine, if_exists="append", index=False)

322

In [84]:
#fact_aum

df = pd.read_csv("../data/processed/03_aum_by_fund_house_cleaned.csv")

df = df.rename(columns={"date" : "aum_date"})

df = df[["fund_house","aum_date", "aum_crore", "num_schemes"]]

df.to_sql("fact_aum", engine, if_exists="append", index=False)

90

In [ ]:
#fact_sip_industry

df = pd.read_csv("../data/processed/04_monthly_sip_inflows_cleaned.csv")

df = df.rename(columns={"active_sip_accounts_crore" : "sip_accounts_crore"})

df = df[["sip_inflow_crore", "sip_accounts_crore", "month"]]

df.to_sql("fact_sip_industry", engine, if_exists="append", index=False)

48

In [99]:
conn = sqlite3.connect('../data/db/bluestock_mf.db')

conn.executescript("ALTER TABLE fact_sip_industry ADD COLUMN yoy_growth_pct REAL; DELETE FROM fact_sip_industry;")

conn.commit()
conn.close()

In [100]:

df = pd.read_csv("../data/processed/04_monthly_sip_inflows_cleaned.csv")

df = df.rename(columns={"active_sip_accounts_crore" : "sip_accounts_crore"})

df = df[["sip_inflow_crore", "sip_accounts_crore", "month", "yoy_growth_pct"]]

df.to_sql("fact_sip_industry", engine, if_exists="append", index=False)

48

In [95]:
#fact_transaction

df = pd.read_csv("../data/processed/08_investor_transactions_cleaned.csv")

df = df[["tx_id", "investor_id", "amfi_code", "transaction_date", "amount_inr", "transaction_type"]]

df.to_sql("fact_transaction", engine, if_exists="append", index=False)

32778

In [101]:
conn = sqlite3.connect('../data/db/bluestock_mf.db')

conn.executescript("ALTER TABLE fact_transaction ADD COLUMN state TEXT; DELETE FROM fact_transaction;")

conn.commit()
conn.close()

In [103]:
conn = sqlite3.connect('../data/db/bluestock_mf.db')

conn.executescript("ALTER TABLE fact_transaction ADD COLUMN payment_mode TEXT; DELETE FROM fact_transaction;")

conn.commit()
conn.close()

In [104]:
#fact_transaction

df = pd.read_csv("../data/processed/08_investor_transactions_cleaned.csv")

df = df[["tx_id", "investor_id", "amfi_code", "transaction_date", "amount_inr", "transaction_type", "state", "payment_mode"]]

df.to_sql("fact_transaction", engine, if_exists="append", index=False)

32778

In [98]:
#verify 

from sqlalchemy import text

with engine.connect() as conn:

    tables = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table'"))
    
    for t in tables:
        print(f"Table Found: {t[0]}")


Table Found: dim_fund
Table Found: dim_date
Table Found: fact_nav
Table Found: sqlite_sequence
Table Found: fact_transaction
Table Found: fact_performance
Table Found: fact_portfolio
Table Found: fact_aum
Table Found: fact_sip_industry


# Running Queries

In [113]:
import sqlite3

conn = sqlite3.connect('../data/db/bluestock_mf.db')

with open("../sql/queries.sql", "r") as f:
    queries = f.read()

queries = queries.split(';')

In [114]:
for query in queries:

    query = query.strip()

    if query:

        print("=" * 80)
        print(query)

        df = pd.read_sql(query, conn)

        print(df)

-- #Query 1 Top 5 funds by AUM
SELECT fund_house, aum_crore FROM fact_aum
ORDER BY aum_crore DESC
LIMIT 5
            fund_house  aum_crore
0      SBI Mutual Fund  1250000.0
1      SBI Mutual Fund  1250000.0
2      SBI Mutual Fund  1114000.0
3      SBI Mutual Fund  1080000.0
4  ICICI Prudential MF  1074000.0
-- #Query 2 Average NAV per month
SELECT amfi_code,strftime('%Y-%m', date) AS month, ROUND(AVG(nav),2) AS avg_nav
FROM fact_nav
GROUP BY amfi_code, strftime('%Y-%m', date)
ORDER BY amfi_code, month
     amfi_code    month  avg_nav
0       100016  2022-01   511.92
1       100016  2022-02   514.54
2       100016  2022-03   522.29
3       100016  2022-04   526.11
4       100016  2022-05   504.34
...        ...      ...      ...
2115    149324  2026-01   253.41
2116    149324  2026-02   249.45
2117    149324  2026-03   263.09
2118    149324  2026-04   305.80
2119    149324  2026-05   304.87

[2120 rows x 3 columns]
-- #Query 3 SIP inflow YoY growth
SELECT month, sip_inflow_crore, yoy_g